# 9. Прогнозная модель: базовые решения
Задача: предсказать, попадёт ли команда в топ-10% приватного лидерборда,
по признакам, известным в начале соревнования. Здесь — простые решения, с которыми будем сравнивать бустинг.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.build_dataset import get_dataset, FEATURES, TARGET
from src.modeling import split_by_competition, evaluate, baseline_models

df = get_dataset()  # при первом запуске собирается долго, потом берётся из data/processed/
df.shape

In [ ]:
df[TARGET].mean(), df["CompetitionId"].nunique()

Разбиваем по соревнованиям: модель должна работать на новых соревнованиях, а не на знакомых.

In [ ]:
train, test = split_by_competition(df)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
len(train), len(test)

## Базовые решения
- `constant` — всегда доля топ-10% из train (ROC-AUC = 0.5 по определению);
- `rule_experience` — без обучения: чем опытнее самый опытный участник, тем выше шанс;
- `logreg` — логистическая регрессия на всех признаках.

In [ ]:
results = {"rule_experience": evaluate(y_test, X_test["max_prev_comps"].fillna(0))}
for name, model in baseline_models().items():
    model.fit(X_train, y_train)
    results[name] = evaluate(y_test, model.predict_proba(X_test)[:, 1])

results = pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)
results

## Коэффициенты логистической регрессии
Признаки стандартизованы, поэтому коэффициенты можно сравнивать между собой.

In [ ]:
logreg = baseline_models()["logreg"].fit(X_train, y_train)
pd.Series(logreg[-1].coef_[0], index=FEATURES).sort_values()

TODO: сравнить с LightGBM (ноутбук 10).